### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [ ]:
# %pip uninstall huggingface-hub transformers fsspec accelerate sentencepiece optimum peft bitsandbytes

In [1]:
%pip --quiet install huggingface-hub transformers fsspec accelerate sentencepiece optimum peft bitsandbytes datasets

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
model_name = 'Enoch/llama-7b-hf'

In [3]:
# loading Llama tokenizer ...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


In [4]:
# ... and the model itself
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

config.json:   0%|          | 0.00/511 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


pytorch_model.bin.index.json:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

pytorch_model-00001-of-00033.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

...

pytorch_model-00031-of-00033.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

pytorch_model-00032-of-00033.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

pytorch_model-00033-of-00033.bin:   0%|          | 0.00/524M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

### Prompt tuning: the story of a fox (2 pts)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: <s>A quick brown fox jumps over the lazy dog.
A quick


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [ ]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:] # cut <s> from start

loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)
outputs.logits.size(), outputs.logits[:, :-1].size()

Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [ ]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)
        # self.embs = nn.Embedding(word_embeddings., word_embeddings.embedding_dim)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
        # This is because we will prepend :num_prompts: padding tokens at the beginning

        # After you are done, you must produce a word embedding vector for each token in input_ids,
        # except that the first :num_prompts: vectors should equal learnable_prompts;
        # any additional vectors after first :num_prompts: ones should be embedded as usual
        # Note: since you're dealing with trainable params, please torch.cat instead of item assignment

        actual_input_ids = input_ids[:, self.num_prompts:]

        embs = self.original_word_embeddings(actual_input_ids)

        embs = torch.cat([self.learnable_prompts, embs], dim=1)

        assert embs.size(0) == input_ids.size(0) and embs.size(1) == input_ids.size(1) and embs.size(2) == model.config.hidden_size, embs.size()

        return embs

In [ ]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


<ipython-input-24-acb6260f9ffb>:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [ ]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [ ]:
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
# batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
# space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
#                                dtype=torch.int64, device=device)
# batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
# batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

# for i in range(100):
#   outputs = model(**batch)
#   next_word_logits = outputs.logits[:, num_prompts : -1, :]
#   true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
#   loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

#   opt.zero_grad()
#   loss.backward()
#   opt.step()

#   print('step', i + 1)
#   print("Loss:", loss.item(), '\n')

# # raise NotImplemented("Your task: iteratively train the model to reduce loss using prompt optimizer (opt)")


# assert loss.item() <= 0.1
# print("Good job!")

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Using HuggingFace PEFT (2 points)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [ ]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
num_virtual_tokens = 16
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"

batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(100):
  logits = model(**batch)['logits']

  next_word_logits: torch.Tensor = logits[:, num_virtual_tokens : -1, :]
  true_next_tokens = batch['input_ids'][:, 1:]

  loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

  opt.zero_grad()
  loss.backward()
  opt.step()

print('final loss', loss.item())
assert loss.item() < 0.1

final loss 0.010847778059542179


In [ ]:
batch = tokenizer('A quick brown fox', return_tensors='pt', return_token_type_ids=False).to(device)
logits = model.generate(**batch)

In [ ]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, :].cpu().numpy().tolist()))


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [ ]:
# re-load the model to remove any previous PEFT tuners
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [5]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))#module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))#module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        module_outputs = self.module(input).to(self.module.weight.device)

        lora_outputs = input.to(self.module.weight.device) @ self.adapter_A @ self.adapter_B

        return module_outputs + lora_outputs

In [ ]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [ ]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B

In [ ]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

<ipython-input-59-f8c92fc392b8>:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [24]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs', report_to=None),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

### Final task: *actually* train the model (4 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

In [6]:
from datasets import load_dataset

ds = load_dataset("codeparrot/codeparrot-clean-valid")

README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


file-000000000054.json.gz:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61373 [00:00<?, ? examples/s]

In [7]:
MAX_LENGTH = 512
def preprocess_function(sample):
    result = tokenizer(
        sample['content'], padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    return result

ds_preprocessed = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/61373 [00:00<?, ? examples/s]

In [13]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

In [14]:
model.lm_head = LoRALayer(model.lm_head, rank=lora_rank).to(device)

In [15]:
from transformers import TrainingArguments

model._hf_peft_config_loaded = True  # silence a warning from HF trainer

batch_size = 4

training_args = TrainingArguments(
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=batch_size, # note: if you want larger batch size, increase gradient_accumulation_steps
    warmup_steps=250,
    max_steps=500,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    output_dir='outputs',
    report_to=None
)

In [17]:
ds_preprocessed

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'path', 'copies', 'size', 'content', 'license', 'hash', 'line_mean', 'line_max', 'alpha_frac', 'autogenerated', 'input_ids', 'attention_mask'],
        num_rows: 61373
    })
})

In [18]:
trainer = transformers.Trainer(
    model=model,
    train_dataset=ds_preprocessed['train'],
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/bitsandbytes/nn/modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


Step,Training Loss
1,1.048600
2,1.134100
3,1.048900
4,1.292700
5,0.969800
6,1.195800
7,1.218700
8,1.130300
9,1.051000
10,1.116800


UnboundLocalError: cannot access local variable 'active_adapters' where it is not associated with a value

In [21]:
prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

generated_list = []
for prompt in prompts:
  batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
  out = model.generate(**batch, max_new_tokens=64, do_sample=True, top_p=0.9)
  generated = tokenizer.decode(out[0][batch['input_ids'].shape[1]:].cpu())
  generated_list.append(generated)

generated_list

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


["from setuptools import setup\n\nwith open('README.rst') as f:\n    long_description = f.read()\n\nsetup(\n    name='pydoc2slides',\n    version='0.2.2',\n    description='Convert existing documentation to slides",
 'random\nfrom time import sleep\nfrom datetime import datetime, timedelta\nfrom dateutil.parser import parse as dateparser\n\nfrom pytg import (\n    TG,\n    TGError,\n    EventTG,\n    EventTGType,\n    TimerEventTG',
 '__future__ import absolute_import, division, print_function, unicode_literals\n\nimport copy\nimport os\nimport platform\n\nfrom setuptools import setup, Extension, Feature\nfrom setuptools.dist import Distribution\n\nfrom numba import __version__, __author',
 'not True:\n    try:\n        userInput = int(input("> "))\n    except ValueError:\n        userInput = input("> ").strip()\n    if userInput in range(1000, 1003):\n        answer = 100',
 ':\n    import json\nexcept ImportError:\n    from django.utils import simplejson as json\n\nfrom django.conf i

In [7]:
generated_list = ["from setuptools import setup\n\nwith open('README.rst') as f:\n    long_description = f.read()\n\nsetup(\n    name='pydoc2slides',\n    version='0.2.2',\n    description='Convert existing documentation to slides",
 'random\nfrom time import sleep\nfrom datetime import datetime, timedelta\nfrom dateutil.parser import parse as dateparser\n\nfrom pytg import (\n    TG,\n    TGError,\n    EventTG,\n    EventTGType,\n    TimerEventTG',
 '__future__ import absolute_import, division, print_function, unicode_literals\n\nimport copy\nimport os\nimport platform\n\nfrom setuptools import setup, Extension, Feature\nfrom setuptools.dist import Distribution\n\nfrom numba import __version__, __author',
 'not True:\n    try:\n        userInput = int(input("> "))\n    except ValueError:\n        userInput = input("> ").strip()\n    if userInput in range(1000, 1003):\n        answer = 100',
 ':\n    import json\nexcept ImportError:\n    from django.utils import simplejson as json\n\nfrom django.conf import settings\nfrom django.contrib.auth.models import User\nfrom django.db import models\nfrom django.db.models.loading import cache\nfrom django.utils import',
 "( ! defined( 'WP_USE_THIS_FILE' ) ) {\n\theader('HTTP/1.1 403 Forbidden');\n\tdie();\n}\n\ndefine( 'WP_USE_THEMES', true );\ndefine( 'WP_USE_TW",
 '(j in 1:4) {\n\n  var i = i + 1\n  for (k in 1:j) {\n\n    var j = j + 1\n    var l = l + 1\n\n    val x = i*i + j*j +',
 ".manuals = {\n  'help': [[\n      #\n      # Help\n      #\n\n      _MNU_SRC_DIR_ = os.path.abspath(torch.__file__)\n      _PYTHON_SOURCE = torch.__file__[:"]

print(generated_list[0])

from setuptools import setup

with open('README.rst') as f:
    long_description = f.read()

setup(
    name='pydoc2slides',
    version='0.2.2',
    description='Convert existing documentation to slides


In [5]:
prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# <A WHOLE LOT OF YOUR CODE>
# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

generated_pre_tuning = []
for prompt in prompts:
  batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
  out = model.generate(**batch, max_new_tokens=64, do_sample=True, top_p=0.9)
  generated = tokenizer.decode(out[0][batch['input_ids'].shape[1]:].cpu())
  generated_pre_tuning.append(generated)

generated_pre_tuning

['○■・・・・・・・ 【＝＝＝＝＝室＝室＝室・室の便物類・ ・・・・・ ◎',
 'collections\nfrom typing import List\n\nfrom .constants import CACHED\nfrom .utils import Cached, cache, cached_property\n\n\nclass Group(object):\n    def __init__(self, name:str):\n        self.name = name\n        self.members = List',
 '__future__ import absolute_import\nfrom __future__ import print_function\nfrom __future__ import unicode_literals\n\nfrom .constants import DRIVERS, INPUT_TYPES, OUTPUT_TYPES\nfrom .constants import DRIVERS',
 '(1)\nwhile ( 1 )\nwhile(1);\nThe “do” in the first three lines is just a syntactic sugar. They are all equivalent.\nwhile(1) is also equivalent to:\nwhile(0);\nwhile(1) or while(0) is also',
 ' /catch are two different things\nSorry, I didn\'t mean to confuse you.\nI just mean, when I say "catching" I mean that I would like to catch that exception as an error value.</s>',
 "i would've seen this on the subway, i would've died, i would've died.\nthat dress, those shoes, that bag.\nthe outfit on the lef

In [8]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for i, prompt in enumerate(prompts):
    # replace placeholders in the format() arguments
    rows.append(row_template.format(prompt, generated_pre_tuning[i], generated_list[i]))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,○■・・・・・・・ 【＝＝＝＝＝室＝室＝室・室の便物類・ ・・・・・ ◎,"from setuptools import setup with open('README.rst') as f: long_description = f.read() setup( name='pydoc2slides', version='0.2.2', description='Convert existing documentation to slides"
`import`,"collections from typing import List from .constants import CACHED from .utils import Cached, cache, cached_property class Group(object): def __init__(self, name:str): self.name = name self.members = List","random from time import sleep from datetime import datetime, timedelta from dateutil.parser import parse as dateparser from pytg import ( TG, TGError, EventTG, EventTGType, TimerEventTG"
`from`,"__future__ import absolute_import from __future__ import print_function from __future__ import unicode_literals from .constants import DRIVERS, INPUT_TYPES, OUTPUT_TYPES from .constants import DRIVERS","__future__ import absolute_import, division, print_function, unicode_literals import copy import os import platform from setuptools import setup, Extension, Feature from setuptools.dist import Distribution from numba import __version__, __author"
`while`,(1) while ( 1 ) while(1); The “do” in the first three lines is just a syntactic sugar. They are all equivalent. while(1) is also equivalent to: while(0); while(1) or while(0) is also,"not True: try: userInput = int(input(""> "")) except ValueError: userInput = input(""> "").strip() if userInput in range(1000, 1003): answer = 100"
`try`,"/catch are two different things Sorry, I didn't mean to confuse you. I just mean, when I say ""catching"" I mean that I would like to catch that exception as an error value.",: import json except ImportError: from django.utils import simplejson as json from django.conf import settings from django.contrib.auth.models import User from django.db import models from django.db.models.loading import cache from django.utils import
`if`,"i would've seen this on the subway, i would've died, i would've died. that dress, those shoes, that bag. the outfit on the left (brown suit + brown bag), oh my god. i died. i died again. i would've","( ! defined( 'WP_USE_THIS_FILE' ) ) { header('HTTP/1.1 403 Forbidden'); die(); } define( 'WP_USE_THEMES', true ); define( 'WP_USE_TW"
`for`,"your event or venue. For the most competitive rate on Inflatable Arches, Inflatable Obstacle Courses, Interactive Inflatable Games, and Inflatable Slides for sale, we’re your source. To browse our inventory of Interactive Inflatable",(j in 1:4) { var i = i + 1 for (k in 1:j) { var j = j + 1 var l = l + 1 val x = i*i + j*j +
`torch`,"and by using a screwdriver as a poker, but it is not clear that he is really making the kind of puncture that might actually have killed him. In any case, the wound must have caused him a lot of pain. The stabbed man is described as ‘f",.manuals = { 'help': [[ # # Help # _MNU_SRC_DIR_ = os.path.abspath(torch.__file__) _PYTHON_SOURCE = torch.__file__[:


If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.